In [ ]:
# @title 1.1 🔍 Check GPU
import torch

print("🔍 GPU Check:")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / 1024**3
        print(f"   GPU {i}: {props.name} ({mem_gb:.1f} GB)")
else:
    print("❌ GPU not found!")

In [ ]:
# @title 1.2 📦 Clone Repository & Install Dependencies
import os
import sys

REPO_URL = "https://github.com/ngnam1104/TriMedAgent.git"
WORK_DIR = "/kaggle/working/TriMedAgent"

if not os.path.exists(WORK_DIR):
    print("📥 Cloning TriMedAgent...")
    !git clone {REPO_URL} {WORK_DIR}
else:
    print("✅ Repository exists.")

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

# Install dependencies (standard transformers + peft, NOT unsloth)
print("\n📦 Installing RL training dependencies...")
!pip install -q transformers>=4.36.0
!pip install -q peft>=0.7.0
!pip install -q accelerate>=0.25.0
!pip install -q bitsandbytes>=0.41.0
!pip install -q trl>=0.7.0
!pip install -q datasets huggingface_hub numpy

print("\n✅ Installation Complete!")
print("   Using: transformers + peft (standard training)")

In [ ]:
# @title 1.3 🔑 Setup HuggingFace Token
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("✅ Logged in to HuggingFace!")
except Exception as e:
    print("⚠️ Could not get HF_TOKEN from secrets.")
    HF_TOKEN = input("Enter HuggingFace Token: ")
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("✅ Logged in!")

---
## 2️⃣ 📊 Reward Functions

In [ ]:
# @title 2.1 🎯 Define Reward Functions
import json
import re
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

@dataclass
class RewardConfig:
    """Reward weights configuration"""
    iou_weight: float = 0.3
    acc_weight: float = 0.4
    format_weight: float = 0.2
    step_penalty: float = -0.1

class RewardFunction:
    """
    Composite Reward Function for TriMedAgent
    
    R_total = w1*R_IoU + w2*R_Acc + w3*R_Format + w4*R_Step
    """
    
    def __init__(self, config: RewardConfig = None):
        self.config = config or RewardConfig()
    
    def compute_iou(self, box1: List[float], box2: List[float]) -> float:
        """Compute IoU between two boxes [x1,y1,x2,y2]"""
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        
        inter = max(0, x2-x1) * max(0, y2-y1)
        area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
        area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
        union = area1 + area2 - inter
        
        return inter / union if union > 0 else 0
    
    def compute_iou_reward(
        self,
        predicted_boxes: List[List[float]],
        ground_truth_boxes: List[List[float]]
    ) -> float:
        """IoU reward - detection accuracy"""
        if not predicted_boxes or not ground_truth_boxes:
            return 0.0
        
        ious = []
        for gt_box in ground_truth_boxes:
            best_iou = max(self.compute_iou(pred, gt_box) for pred in predicted_boxes)
            ious.append(best_iou)
        
        return np.mean(ious)
    
    def compute_format_reward(self, response: str) -> float:
        """Format reward - valid JSON structure"""
        try:
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                score = 0.0
                if 'thought' in data:
                    score += 0.4
                if 'action' in data:
                    score += 0.4
                if 'action_input' in data:
                    score += 0.2
                return score
        except:
            pass
        return 0.0
    
    def compute_accuracy_reward(
        self,
        predicted_answer: str,
        ground_truth_answer: str
    ) -> float:
        """Answer accuracy reward"""
        if not predicted_answer or not ground_truth_answer:
            return 0.5  # Neutral
        
        pred_lower = predicted_answer.lower()
        gt_lower = ground_truth_answer.lower()
        
        # Exact match
        if pred_lower == gt_lower:
            return 1.0
        
        # Partial match (keyword overlap)
        pred_words = set(pred_lower.split())
        gt_words = set(gt_lower.split())
        overlap = len(pred_words & gt_words) / max(len(gt_words), 1)
        
        return min(1.0, overlap)
    
    def compute_step_reward(self, num_steps: int) -> float:
        """Step efficiency reward (penalize too many steps)"""
        if num_steps <= 2:
            return 0.1
        elif num_steps <= 4:
            return 0.0
        else:
            return self.config.step_penalty * (num_steps - 4)
    
    def compute(
        self,
        response: str,
        predicted_boxes: List[List[float]] = None,
        ground_truth_boxes: List[List[float]] = None,
        predicted_answer: str = None,
        ground_truth_answer: str = None,
        num_steps: int = 1
    ) -> Dict[str, float]:
        """Compute total reward"""
        
        r_iou = self.compute_iou_reward(
            predicted_boxes or [], ground_truth_boxes or []
        )
        r_format = self.compute_format_reward(response)
        r_acc = self.compute_accuracy_reward(
            predicted_answer or "", ground_truth_answer or ""
        )
        r_step = self.compute_step_reward(num_steps)
        
        total = (
            self.config.iou_weight * r_iou +
            self.config.acc_weight * r_acc +
            self.config.format_weight * r_format +
            r_step
        )
        
        return {
            'total': total,
            'iou': r_iou,
            'format': r_format,
            'accuracy': r_acc,
            'step': r_step
        }

reward_fn = RewardFunction()
print("✅ Reward functions defined!")
print(f"   Weights: IoU={reward_fn.config.iou_weight}, Acc={reward_fn.config.acc_weight}, Format={reward_fn.config.format_weight}")

In [ ]:
# @title 2.2 🧪 Test Reward Function
# Test example
test_response = '{"thought": "Detecting nodules", "action": "GroundingDINO", "action_input": {"prompt": "lung nodule"}}'
test_pred_boxes = [[100, 100, 200, 200]]
test_gt_boxes = [[110, 110, 210, 210]]

reward = reward_fn.compute(
    response=test_response,
    predicted_boxes=test_pred_boxes,
    ground_truth_boxes=test_gt_boxes,
    predicted_answer="nodule detected",
    ground_truth_answer="lung nodule",
    num_steps=2
)

print("📊 Test Reward:")
for k, v in reward.items():
    print(f"   {k}: {v:.4f}")

---
## 3️⃣ 📊 Download & Prepare RL Dataset

**RL Training cần Ground Truth để tính Reward:**
- **IoU Reward**: So sánh predicted boxes vs ground truth boxes
- **Accuracy Reward**: So sánh answer với expert annotation
- **Format Reward**: JSON validity
- **Step Penalty**: Penalize quá nhiều steps

**Data Sources:**
- **VinDr-CXR**: 18,000 X-rays với bounding boxes (Kaggle)
- **RSNA Pneumonia**: 26,684 images với detection boxes (Kaggle)
- **Object-CXR**: Chest X-ray với foreign object detection

In [ ]:
# @title 3.1 📥 Download VinDr-CXR (Detection Dataset)
import os
import json
from pathlib import Path

DATA_DIR = Path("data")
RL_DIR = DATA_DIR / "rl_dataset"
RAW_DIR = DATA_DIR / "raw"
RL_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("="*60)
print("📥 DOWNLOADING VinDr-CXR (Detection with Bounding Boxes)")
print("="*60)

VINDR_DIR = RAW_DIR / "vindr-cxr"

print("""
📋 VinDr-CXR Dataset Info:
   - 18,000 chest X-ray images
   - 15 disease classes with bounding boxes
   - Classes: Aortic enlargement, Atelectasis, Calcification, 
     Cardiomegaly, Consolidation, ILD, Infiltration, Lung Opacity,
     Nodule/Mass, Pleural effusion, Pleural thickening, Pneumothorax,
     Pulmonary fibrosis, Other lesion, No finding
""")

if not VINDR_DIR.exists():
    print("📥 Downloading from Kaggle...")
    print("   Competition: vinbigdata-chest-xray-abnormalities-detection")
    
    try:
        !pip install -q kaggle
        
        # Check Kaggle credentials
        kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
        if not kaggle_json.exists():
            print("\n⚠️ Kaggle credentials required!")
            try:
                from kaggle_secrets import UserSecretsClient
                secrets = UserSecretsClient()
                kaggle_user = secrets.get_secret("KAGGLE_USERNAME")
                kaggle_key = secrets.get_secret("KAGGLE_KEY")
                
                !mkdir -p ~/.kaggle
                with open(kaggle_json, 'w') as f:
                    json.dump({"username": kaggle_user, "key": kaggle_key}, f)
                !chmod 600 ~/.kaggle/kaggle.json
                print("✅ Kaggle credentials loaded from secrets!")
            except:
                print("   Add KAGGLE_USERNAME and KAGGLE_KEY to Kaggle Secrets")
                print("   Or download manually from:")
                print("   https://www.kaggle.com/c/vinbigdata-chest-xray-abnormalities-detection/data")
        
        # Download competition data
        !kaggle competitions download -c vinbigdata-chest-xray-abnormalities-detection -p {RAW_DIR}
        
        # Unzip
        vindr_zip = RAW_DIR / "vinbigdata-chest-xray-abnormalities-detection.zip"
        if vindr_zip.exists():
            print("📦 Extracting...")
            !unzip -q {vindr_zip} -d {VINDR_DIR}
            print("✅ VinDr-CXR downloaded!")
            
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\n📋 Manual download:")
        print("   1. Accept competition rules at:")
        print("      https://www.kaggle.com/c/vinbigdata-chest-xray-abnormalities-detection/rules")
        print("   2. Download train.csv and images")
        print("   3. Extract to: data/raw/vindr-cxr/")
else:
    print("✅ VinDr-CXR already exists!")

# Show contents
if VINDR_DIR.exists():
    print(f"\n📁 Contents of {VINDR_DIR}:")
    for item in list(VINDR_DIR.iterdir())[:10]:
        print(f"   - {item.name}")

In [ ]:
# @title 3.2 📥 Download RSNA Pneumonia Detection Dataset
print("="*60)
print("📥 DOWNLOADING RSNA PNEUMONIA DETECTION")
print("="*60)

RSNA_DIR = RAW_DIR / "rsna-pneumonia"

print("""
📋 RSNA Pneumonia Detection Info:
   - 26,684 chest X-ray images  
   - Binary classification + bounding boxes
   - Labels: Normal, Lung Opacity (with boxes)
   - Perfect for IoU reward training
""")

if not RSNA_DIR.exists():
    print("📥 Downloading from Kaggle...")
    
    try:
        # Download competition data
        !kaggle competitions download -c rsna-pneumonia-detection-challenge -p {RAW_DIR}
        
        # Unzip
        rsna_zip = RAW_DIR / "rsna-pneumonia-detection-challenge.zip"
        if rsna_zip.exists():
            print("📦 Extracting...")
            !unzip -q {rsna_zip} -d {RSNA_DIR}
            print("✅ RSNA Pneumonia downloaded!")
            
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\n📋 Manual download:")
        print("   1. Go to: https://www.kaggle.com/c/rsna-pneumonia-detection-challenge/data")
        print("   2. Download stage_2_train_labels.csv and images")
        print("   3. Extract to: data/raw/rsna-pneumonia/")
else:
    print("✅ RSNA Pneumonia already exists!")

if RSNA_DIR.exists():
    print(f"\n📁 Contents of {RSNA_DIR}:")
    for item in list(RSNA_DIR.iterdir())[:10]:
        print(f"   - {item.name}")

In [ ]:
# @title 3.3 🔄 Preprocess VinDr-CXR → RL Format
import pandas as pd
import json
from pathlib import Path

print("="*60)
print("🔄 PREPROCESSING VinDr-CXR → RL Format")
print("="*60)

# Disease class mapping
VINDR_CLASSES = {
    0: "Aortic enlargement",
    1: "Atelectasis",
    2: "Calcification", 
    3: "Cardiomegaly",
    4: "Consolidation",
    5: "ILD",
    6: "Infiltration",
    7: "Lung Opacity",
    8: "Nodule/Mass",
    9: "Other lesion",
    10: "Pleural effusion",
    11: "Pleural thickening",
    12: "Pneumothorax",
    13: "Pulmonary fibrosis",
    14: "No finding"
}

# Question templates for each class
QUESTION_TEMPLATES = {
    "Aortic enlargement": "Có dấu hiệu phình động mạch chủ không?",
    "Atelectasis": "Kiểm tra xẹp phổi (atelectasis) trong ảnh.",
    "Calcification": "Tìm các vùng vôi hóa trong phổi.",
    "Cardiomegaly": "Đánh giá kích thước tim và phát hiện cardiomegaly.",
    "Consolidation": "Có vùng đông đặc (consolidation) không?",
    "ILD": "Kiểm tra bệnh phổi kẽ (ILD).",
    "Infiltration": "Tìm các vùng thâm nhiễm trong phổi.",
    "Lung Opacity": "Detect các vùng mờ bất thường trong phổi.",
    "Nodule/Mass": "Tìm các nốt mờ hoặc khối u trong phổi.",
    "Other lesion": "Phát hiện các tổn thương khác trong ảnh.",
    "Pleural effusion": "Có tràn dịch màng phổi không?",
    "Pleural thickening": "Kiểm tra dày màng phổi.",
    "Pneumothorax": "Phát hiện tràn khí màng phổi (pneumothorax).",
    "Pulmonary fibrosis": "Có dấu hiệu xơ phổi không?",
}

def process_vindr_for_rl(csv_path: Path, image_dir: Path = None) -> list:
    """Process VinDr-CXR annotations for RL training"""
    examples = []
    
    if not csv_path.exists():
        print(f"⚠️ File not found: {csv_path}")
        return examples
    
    print(f"📖 Reading: {csv_path.name}")
    df = pd.read_csv(csv_path)
    print(f"   Total rows: {len(df)}")
    
    # Group by image
    grouped = df.groupby('image_id')
    
    for image_id, group in grouped:
        # Skip "No finding" images for RL (no boxes)
        if len(group) == 1 and group.iloc[0]['class_id'] == 14:
            continue
        
        # Get all boxes for this image
        boxes = []
        classes = []
        for _, row in group.iterrows():
            if row['class_id'] != 14 and pd.notna(row.get('x_min')):
                boxes.append([
                    float(row['x_min']),
                    float(row['y_min']),
                    float(row['x_max']),
                    float(row['y_max'])
                ])
                classes.append(VINDR_CLASSES.get(row['class_id'], 'unknown'))
        
        if not boxes:
            continue
        
        # Create example for each class
        for cls in set(classes):
            cls_boxes = [b for b, c in zip(boxes, classes) if c == cls]
            question = QUESTION_TEMPLATES.get(cls, f"Detect {cls} in the image.")
            
            # Determine difficulty based on number of boxes and class
            if len(cls_boxes) == 1:
                difficulty = "easy"
            elif len(cls_boxes) <= 3:
                difficulty = "medium"
            else:
                difficulty = "hard"
            
            example = {
                "prompt": question,
                "image": f"{image_id}.dicom",
                "ground_truth_boxes": cls_boxes,
                "ground_truth_answer": f"{cls} detected: {len(cls_boxes)} region(s)",
                "class_name": cls,
                "difficulty": difficulty
            }
            examples.append(example)
    
    return examples

# Process VinDr-CXR
vindr_examples = []

# Try different possible paths
possible_csvs = [
    VINDR_DIR / "train.csv",
    VINDR_DIR / "train_annotations.csv",
    RAW_DIR / "train.csv",
]

for csv_path in possible_csvs:
    if csv_path.exists():
        vindr_examples = process_vindr_for_rl(csv_path)
        break

if vindr_examples:
    print(f"\n✅ VinDr-CXR: {len(vindr_examples)} RL examples")
    
    # Stats
    from collections import Counter
    difficulties = Counter(ex['difficulty'] for ex in vindr_examples)
    classes = Counter(ex['class_name'] for ex in vindr_examples)
    
    print(f"\n📊 Difficulty Distribution:")
    for diff, count in difficulties.most_common():
        print(f"   - {diff}: {count}")
    
    print(f"\n📊 Top Classes:")
    for cls, count in classes.most_common(5):
        print(f"   - {cls}: {count}")
else:
    print("⚠️ VinDr-CXR not processed. Will use sample data.")

In [ ]:
# @title 3.4 🔄 Preprocess RSNA Pneumonia → RL Format
import pandas as pd
from pathlib import Path

print("="*60)
print("🔄 PREPROCESSING RSNA PNEUMONIA → RL Format")
print("="*60)

def process_rsna_for_rl(csv_path: Path) -> list:
    """Process RSNA Pneumonia annotations for RL training"""
    examples = []
    
    if not csv_path.exists():
        print(f"⚠️ File not found: {csv_path}")
        return examples
    
    print(f"📖 Reading: {csv_path.name}")
    df = pd.read_csv(csv_path)
    print(f"   Total rows: {len(df)}")
    
    # Filter only positive cases (with boxes)
    positive_df = df[df['Target'] == 1].dropna(subset=['x', 'y', 'width', 'height'])
    print(f"   Positive cases with boxes: {len(positive_df)}")
    
    # Group by patient
    grouped = positive_df.groupby('patientId')
    
    for patient_id, group in grouped:
        boxes = []
        for _, row in group.iterrows():
            # Convert x, y, width, height to x1, y1, x2, y2
            x1 = float(row['x'])
            y1 = float(row['y'])
            x2 = x1 + float(row['width'])
            y2 = y1 + float(row['height'])
            boxes.append([x1, y1, x2, y2])
        
        if not boxes:
            continue
        
        # Difficulty based on number of boxes
        if len(boxes) == 1:
            difficulty = "easy"
        elif len(boxes) == 2:
            difficulty = "medium"
        else:
            difficulty = "hard"
        
        # Question templates
        questions = [
            "Kiểm tra dấu hiệu viêm phổi trong ảnh X-quang.",
            "Detect pneumonia regions in this chest X-ray.",
            "Tìm các vùng đông đặc do viêm phổi.",
            "Is there any lung opacity indicating pneumonia?",
            "Phát hiện và locate vùng viêm phổi.",
        ]
        
        import random
        question = random.choice(questions)
        
        example = {
            "prompt": question,
            "image": f"{patient_id}.dcm",
            "ground_truth_boxes": boxes,
            "ground_truth_answer": f"Pneumonia detected: {len(boxes)} region(s)",
            "class_name": "pneumonia",
            "difficulty": difficulty
        }
        examples.append(example)
    
    return examples

# Process RSNA
rsna_examples = []

possible_csvs = [
    RSNA_DIR / "stage_2_train_labels.csv",
    RSNA_DIR / "train_labels.csv",
    RAW_DIR / "stage_2_train_labels.csv",
]

for csv_path in possible_csvs:
    if csv_path.exists():
        rsna_examples = process_rsna_for_rl(csv_path)
        break

if rsna_examples:
    print(f"\n✅ RSNA Pneumonia: {len(rsna_examples)} RL examples")
    
    from collections import Counter
    difficulties = Counter(ex['difficulty'] for ex in rsna_examples)
    print(f"\n📊 Difficulty Distribution:")
    for diff, count in difficulties.most_common():
        print(f"   - {diff}: {count}")
else:
    print("⚠️ RSNA not processed. Will use sample data.")

In [ ]:
# @title 3.5 📦 Combine & Save RL Dataset
import json
import random
from pathlib import Path
from collections import Counter

print("="*60)
print("📦 COMBINING ALL DATA → RL DATASET")
print("="*60)

# Combine all examples
all_rl_examples = []

# Add VinDr-CXR
if vindr_examples:
    all_rl_examples.extend(vindr_examples)
    print(f"✅ Added VinDr-CXR: {len(vindr_examples)} examples")

# Add RSNA Pneumonia
if rsna_examples:
    all_rl_examples.extend(rsna_examples)
    print(f"✅ Added RSNA Pneumonia: {len(rsna_examples)} examples")

# If no real data, use sample data for demo
if len(all_rl_examples) < 50:
    print("\n⚠️ Insufficient real data. Adding sample data for demo...")
    
    SAMPLE_RL_DATA = [
        {
            "prompt": "Tìm các nốt mờ bất thường trong phổi.",
            "image": None,
            "ground_truth_boxes": [[100, 150, 200, 250], [300, 200, 400, 300]],
            "ground_truth_answer": "lung nodule detected in upper right and lower left regions",
            "difficulty": "easy"
        },
        {
            "prompt": "Phân tích ảnh X-quang và tìm các bất thường.",
            "image": None,
            "ground_truth_boxes": [[50, 100, 150, 200]],
            "ground_truth_answer": "cardiomegaly observed with enlarged cardiac silhouette",
            "difficulty": "medium"
        },
        {
            "prompt": "Kiểm tra dấu hiệu viêm phổi và segment vùng tổn thương.",
            "image": None,
            "ground_truth_boxes": [[200, 100, 350, 280], [180, 300, 320, 420]],
            "ground_truth_answer": "bilateral pneumonia consolidation with ground glass opacities",
            "difficulty": "hard"
        },
        {
            "prompt": "Có tràn dịch màng phổi không?",
            "image": None,
            "ground_truth_boxes": [[50, 400, 200, 512]],
            "ground_truth_answer": "pleural effusion present at right lung base",
            "difficulty": "easy"
        },
        {
            "prompt": "Đánh giá kích thước tim và phát hiện cardiomegaly.",
            "image": None,
            "ground_truth_boxes": [[150, 150, 400, 380]],
            "ground_truth_answer": "cardiomegaly with cardiothoracic ratio > 0.5",
            "difficulty": "medium"
        },
        {
            "prompt": "Detect và segment vùng xẹp phổi (atelectasis).",
            "image": None,
            "ground_truth_boxes": [[250, 200, 380, 350]],
            "ground_truth_answer": "atelectasis in lower lobe",
            "difficulty": "hard"
        },
        {
            "prompt": "Có dấu hiệu gãy xương sườn không?",
            "image": None,
            "ground_truth_boxes": [[120, 180, 180, 220]],
            "ground_truth_answer": "rib fracture at 6th rib lateral aspect",
            "difficulty": "medium"
        },
        {
            "prompt": "Detect lung opacity in this chest X-ray.",
            "image": None,
            "ground_truth_boxes": [[80, 120, 220, 280], [320, 150, 450, 320]],
            "ground_truth_answer": "bilateral lung opacities suggesting pneumonia",
            "difficulty": "medium"
        }
    ]
    
    # Expand
    expanded_samples = SAMPLE_RL_DATA * 20  # 160 samples
    all_rl_examples.extend(expanded_samples)
    print(f"✅ Added sample data: {len(expanded_samples)} examples")

# Shuffle
random.seed(42)
random.shuffle(all_rl_examples)

# Save
output_file = RL_DIR / "train.jsonl"
with open(output_file, 'w', encoding='utf-8') as f:
    for item in all_rl_examples:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"\n" + "="*60)
print("📊 RL DATASET SUMMARY")
print("="*60)
print(f"📁 Output: {output_file}")
print(f"📄 Total samples: {len(all_rl_examples)}")

# Difficulty distribution
difficulties = Counter(ex.get('difficulty', 'unknown') for ex in all_rl_examples)
print(f"\n📊 Difficulty Distribution:")
for diff, count in sorted(difficulties.items()):
    pct = count / len(all_rl_examples) * 100
    print(f"   - {diff}: {count} ({pct:.1f}%)")

# Class distribution
if any('class_name' in ex for ex in all_rl_examples):
    classes = Counter(ex.get('class_name', 'unknown') for ex in all_rl_examples)
    print(f"\n📊 Top Classes:")
    for cls, count in classes.most_common(8):
        print(f"   - {cls}: {count}")

# Boxes stats
total_boxes = sum(len(ex.get('ground_truth_boxes', [])) for ex in all_rl_examples)
avg_boxes = total_boxes / len(all_rl_examples) if all_rl_examples else 0
print(f"\n📊 Bounding Boxes:")
print(f"   - Total boxes: {total_boxes}")
print(f"   - Avg boxes/sample: {avg_boxes:.2f}")

---
## 4️⃣ 🚀 GRPO Training Configuration

In [ ]:
# @title 4.1 ⚙️ GRPO Config
from dataclasses import dataclass, field
from typing import List

@dataclass
class GRPOConfig:
    # Model - LLaVA-Med for medical imaging
    base_model: str = "chaoyinshe/llava-med-v1.5-mistral-7b-hf"
    sft_adapter: str = "ngnam1104/trimedagent-sft-v1"  # From Stage 1
    
    # GRPO hyperparameters
    group_size: int = 4  # Number of responses per prompt
    temperature: float = 0.7
    beta: float = 0.1  # KL penalty
    
    # Training
    max_steps: int = 200
    batch_size: int = 1
    gradient_accumulation_steps: int = 4
    learning_rate: float = 1e-5
    max_new_tokens: int = 256
    
    # Curriculum
    use_curriculum: bool = True
    curriculum_levels: List[str] = field(default_factory=lambda: ['easy', 'medium', 'hard'])
    
    # Output
    output_dir: str = "checkpoints/rl_adapter"
    hub_model_id: str = "ngnam1104/trimedagent-grpo-v1"

config = GRPOConfig()
print("✅ GRPO Config ready!")
print(f"   Base Model: {config.base_model}")
print(f"   SFT Adapter: {config.sft_adapter}")
print(f"   Group Size: {config.group_size}")
print(f"   Max Steps: {config.max_steps}")
print(f"   Curriculum: {config.use_curriculum}")

In [ ]:
# @title 4.2 📚 Load Model with SFT Adapter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

print(f"🚀 Loading base model: {config.base_model}")
print("   (LLaVA-Med - specialized for medical imaging)")

# Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Base model - LLaVA-Med
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)

print(f"📥 Loading SFT adapter: {config.sft_adapter}")
model = PeftModel.from_pretrained(
    base_model,
    config.sft_adapter,
    is_trainable=True
)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

print("✅ LLaVA-Med loaded with SFT adapter!")

In [ ]:
# @title 4.3 📊 Load RL Dataset
from torch.utils.data import Dataset, DataLoader
import json

class RLDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=2048, difficulty_filter=None):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data = []
        
        with open(data_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    item = json.loads(line)
                    if difficulty_filter is None or item.get('difficulty') == difficulty_filter:
                        self.data.append(item)
        
        print(f"Loaded {len(self.data)} RL examples" + 
              (f" (difficulty={difficulty_filter})" if difficulty_filter else ""))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        prompt = f"<s>[INST] {item['prompt']} [/INST]"
        
        encodings = self.tokenizer(
            prompt,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze(),
            'prompt': item['prompt'],
            'ground_truth_boxes': item.get('ground_truth_boxes', []),
            'ground_truth_answer': item.get('ground_truth_answer', ''),
            'difficulty': item.get('difficulty', 'medium')
        }

# Load full dataset
full_dataset = RLDataset("data/rl_dataset/train.jsonl", tokenizer)

---
## 5️⃣ 🔥 GRPO Training Loop

In [ ]:
# @title 5.1 🎯 GRPO Training Functions
from transformers import GenerationConfig
import torch.nn.functional as F
from tqdm import tqdm
from collections import defaultdict

def generate_responses(model, tokenizer, prompt_ids, attention_mask, num_responses, max_new_tokens=256, temperature=0.7):
    """Generate multiple responses for GRPO"""
    model.eval()
    responses = []
    log_probs_list = []
    
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
    
    with torch.no_grad():
        for _ in range(num_responses):
            outputs = model.generate(
                input_ids=prompt_ids,
                attention_mask=attention_mask,
                generation_config=gen_config,
                output_scores=True,
                return_dict_in_generate=True
            )
            
            response_ids = outputs.sequences[0, prompt_ids.shape[1]:]
            response_text = tokenizer.decode(response_ids, skip_special_tokens=True)
            responses.append(response_text)
            
            # Log probs
            if outputs.scores:
                log_probs = []
                for i, scores in enumerate(outputs.scores):
                    probs = F.softmax(scores, dim=-1)
                    token_id = response_ids[i] if i < len(response_ids) else 0
                    log_prob = torch.log(probs[0, token_id] + 1e-8)
                    log_probs.append(log_prob)
                log_probs_list.append(torch.stack(log_probs) if log_probs else torch.zeros(1))
            else:
                log_probs_list.append(torch.zeros(1))
    
    return responses, log_probs_list

def extract_boxes_from_response(response):
    """Extract boxes from JSON response"""
    try:
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            data = json.loads(match.group(0))
            action_input = data.get('action_input', {})
            if 'boxes' in action_input:
                return action_input['boxes']
    except:
        pass
    return []

def count_steps(response):
    """Count action steps in response"""
    return max(1, len(re.findall(r'"action"\s*:', response)))

print("✅ GRPO functions defined!")

In [ ]:
# @title 5.2 🚂 Run GRPO Training
import torch.optim as optim

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.01)

# Metrics
metrics = defaultdict(list)

def grpo_step(batch):
    """Single GRPO training step"""
    prompt_ids = batch['input_ids'].unsqueeze(0).to(model.device)
    attention_mask = batch['attention_mask'].unsqueeze(0).to(model.device)
    gt_boxes = batch['ground_truth_boxes']
    gt_answer = batch['ground_truth_answer']
    
    # Generate responses
    responses, log_probs_list = generate_responses(
        model, tokenizer, prompt_ids, attention_mask,
        config.group_size, config.max_new_tokens, config.temperature
    )
    
    # Compute rewards
    rewards = []
    details = []
    for resp in responses:
        pred_boxes = extract_boxes_from_response(resp)
        num_steps = count_steps(resp)
        
        reward = reward_fn.compute(
            response=resp,
            predicted_boxes=pred_boxes,
            ground_truth_boxes=gt_boxes,
            predicted_answer=resp[:200],
            ground_truth_answer=gt_answer,
            num_steps=num_steps
        )
        rewards.append(reward['total'])
        details.append(reward)
    
    rewards = np.array(rewards)
    rewards_normalized = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
    
    # Policy gradient loss
    model.train()
    total_loss = 0.0
    
    for i, (log_probs, reward) in enumerate(zip(log_probs_list, rewards_normalized)):
        if len(log_probs) > 0 and isinstance(log_probs, torch.Tensor):
            loss = -reward * log_probs.sum()
            total_loss += loss
    
    total_loss = total_loss / config.group_size
    
    # Backward
    optimizer.zero_grad()
    if isinstance(total_loss, torch.Tensor):
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        loss_val = total_loss.item()
    else:
        loss_val = float(total_loss)
    
    return {
        'loss': loss_val,
        'mean_reward': float(rewards.mean()),
        'max_reward': float(rewards.max()),
        'format_reward': np.mean([d['format'] for d in details]),
    }

# Training loop
print("🚂 Starting GRPO Training...")
print(f"   Max steps: {config.max_steps}")
print(f"   Group size: {config.group_size}")
print("="*50)

dataloader = DataLoader(full_dataset, batch_size=1, shuffle=True)
global_step = 0

pbar = tqdm(total=config.max_steps, desc="GRPO Training")

while global_step < config.max_steps:
    for batch in dataloader:
        step_metrics = grpo_step(batch)
        
        for k, v in step_metrics.items():
            metrics[k].append(v)
        
        global_step += 1
        pbar.update(1)
        
        if global_step % 10 == 0:
            avg_loss = np.mean(metrics['loss'][-10:])
            avg_reward = np.mean(metrics['mean_reward'][-10:])
            pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'reward': f'{avg_reward:.4f}'})
        
        if global_step >= config.max_steps:
            break

pbar.close()
print("\n✅ GRPO Training Complete!")

In [ ]:
# @title 5.3 📊 Plot GRPO Training Metrics (for Report)
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Policy Loss
ax1 = axes[0, 0]
ax1.plot(metrics['loss'], alpha=0.3, color='blue', label='Raw Loss')
window = 10
if len(metrics['loss']) >= window:
    smoothed = np.convolve(metrics['loss'], np.ones(window)/window, mode='valid')
    ax1.plot(range(window-1, len(metrics['loss'])), smoothed, 'b-', linewidth=2, label='Smoothed')
ax1.set_xlabel('Training Steps', fontsize=11)
ax1.set_ylabel('Policy Loss', fontsize=11)
ax1.set_title('📉 GRPO Policy Loss', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Reward Curves
ax2 = axes[0, 1]
ax2.plot(metrics['mean_reward'], alpha=0.5, color='green', label='Mean Reward')
ax2.plot(metrics['max_reward'], alpha=0.5, color='orange', label='Max Reward')
if len(metrics['mean_reward']) >= window:
    smoothed_mean = np.convolve(metrics['mean_reward'], np.ones(window)/window, mode='valid')
    ax2.plot(range(window-1, len(metrics['mean_reward'])), smoothed_mean, 'g-', linewidth=2, label='Smoothed Mean')
ax2.set_xlabel('Training Steps', fontsize=11)
ax2.set_ylabel('Reward', fontsize=11)
ax2.set_title('📈 Reward Progress', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Format Reward (JSON validity)
ax3 = axes[1, 0]
ax3.plot(metrics['format_reward'], alpha=0.5, color='purple')
if len(metrics['format_reward']) >= window:
    smoothed_fmt = np.convolve(metrics['format_reward'], np.ones(window)/window, mode='valid')
    ax3.plot(range(window-1, len(metrics['format_reward'])), smoothed_fmt, 'purple', linewidth=2)
ax3.set_xlabel('Training Steps', fontsize=11)
ax3.set_ylabel('Format Reward', fontsize=11)
ax3.set_title('✅ JSON Format Correctness', fontsize=13, fontweight='bold')
ax3.set_ylim([0, 1.1])
ax3.grid(True, alpha=0.3)

# Plot 4: Reward Distribution
ax4 = axes[1, 1]
ax4.hist(metrics['mean_reward'], bins=30, alpha=0.7, color='teal', edgecolor='white')
ax4.axvline(x=np.mean(metrics['mean_reward']), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(metrics["mean_reward"]):.3f}')
ax4.set_xlabel('Reward Value', fontsize=11)
ax4.set_ylabel('Frequency', fontsize=11)
ax4.set_title('📊 Reward Distribution', fontsize=13, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grpo_training_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("📊 GRPO TRAINING SUMMARY")
print("="*60)
print(f"   Total Steps: {len(metrics['loss'])}")
print(f"   Final Loss: {np.mean(metrics['loss'][-20:]):.4f}")
print(f"   Final Mean Reward: {np.mean(metrics['mean_reward'][-20:]):.4f}")
print(f"   Best Reward: {max(metrics['max_reward']):.4f}")
print(f"   Format Accuracy: {np.mean(metrics['format_reward'][-20:])*100:.1f}%")
print("="*60)
print("✅ Metrics saved to: grpo_training_metrics.png")

---
## 6️⃣ 💾 Save & Push to HuggingFace

In [ ]:
# @title 6.1 💾 Save Adapter Locally
from pathlib import Path
import json

output_path = Path(config.output_dir) / "final"
output_path.mkdir(parents=True, exist_ok=True)

# Save model
model.save_pretrained(str(output_path))
tokenizer.save_pretrained(str(output_path))

# Save metrics
with open(output_path / "training_metrics.json", 'w') as f:
    json.dump({k: list(v) for k, v in metrics.items()}, f)

# Save config
config_dict = {
    'base_model': config.base_model,
    'sft_adapter': config.sft_adapter,
    'group_size': config.group_size,
    'max_steps': config.max_steps,
    'learning_rate': config.learning_rate,
}
with open(output_path / "grpo_config.json", 'w') as f:
    json.dump(config_dict, f, indent=2)

print(f"✅ Adapter saved to: {output_path}")

In [ ]:
# @title 6.2 📤 Push to HuggingFace Hub
REPO_ID = config.hub_model_id  # e.g., "ngnam1104/trimedagent-grpo-v1"

print(f"📤 Pushing to HuggingFace: {REPO_ID}")

try:
    model.push_to_hub(REPO_ID, use_auth_token=True)
    tokenizer.push_to_hub(REPO_ID, use_auth_token=True)
    
    print(f"\n✅ Successfully pushed to: https://huggingface.co/{REPO_ID}")
    
except Exception as e:
    print(f"❌ Push failed: {e}")
    print(f"\nTry manual upload:")
    print(f"   huggingface-cli upload {REPO_ID} {output_path}")

In [ ]:
# @title 6.3 📋 Create Model Card
model_card = f"""---
license: apache-2.0
tags:
  - medical
  - vision
  - llava
  - lora
  - grpo
  - reinforcement-learning
  - trimedagent
base_model: {config.base_model}
---

# TriMedAgent GRPO Adapter

LoRA adapter fine-tuned with **Group Relative Policy Optimization (GRPO)** for Medical Visual Agent.

## Training Details

- **Base Model**: `{config.base_model}`
- **SFT Adapter**: `{config.sft_adapter}`
- **Method**: GRPO (DeepSeek-R1 style)
- **Group Size**: {config.group_size}
- **Steps**: {config.max_steps}

## Reward Function

```
R_total = 0.3×R_IoU + 0.4×R_Acc + 0.2×R_Format + 0.1×R_Step
```

- **R_IoU**: Detection accuracy (bounding box overlap)
- **R_Acc**: Answer accuracy
- **R_Format**: JSON structure validity
- **R_Step**: Efficiency penalty

## Usage

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained("{config.base_model}")
model = PeftModel.from_pretrained(base_model, "{REPO_ID}")
```

## Citation

```
@misc{{trimedagent,
  title={{TriMedAgent: Medical Visual Agent with GRPO}},
  author={{Team}},
  year={{2025}}
}}
```
"""

with open(output_path / "README.md", 'w') as f:
    f.write(model_card)

print("✅ Model card created!")
print(f"\n🎉 GRPO Training Pipeline Complete!")
print(f"\n📋 Next: Test with notebook 03_demo.ipynb")

---
## 🎉 Done!

GRPO Training hoàn tất. Model đã được push lên HuggingFace.

**Pipeline hoàn chỉnh:**
1. ✅ `01_sft_training.ipynb` - SFT với LoRA
2. ✅ `02_rl_grpo_training.ipynb` - GRPO Reinforcement Learning
3. ⏭️ `03_demo.ipynb` - Demo với Gradio UI